In [1]:
from src.etl_pipeline import ETLPipeline
from src.utils import create_directories, clean_directories
import time

In [2]:
input_path = "../data/data.csv"

In [3]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

In [4]:
spark = SparkSession.builder.appName("CreditScoreETL")\
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")\
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")\
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(spark).getOrCreate()

:: loading settings :: url = jar:file:/home/ilya/workspace/projects/spark-experiments/lakehouse_toy/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ilya/.ivy2/cache
The jars for the packages stored in: /home/ilya/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-09376700-c38c-43aa-97c7-42c1d0307503;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 201ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   | 

In [5]:
etl = ETLPipeline(spark)

In [6]:
from tqdm.notebook import tqdm

wall_times = []

for _ in tqdm(range(3)):
    create_directories('..')
    start = time.time()
    etl(input_path)
    end = time.time()
    wall_times.append(end - start)
    clean_directories('..')

print(f"Average Wall Time: {(sum(wall_times) / len(wall_times)) * 1000} ms")

  0%|          | 0/3 [00:00<?, ?it/s]

25/06/14 23:50:15 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
25/06/14 23:50:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Average Wall Time: 51848.54078292847 ms


In [10]:
from functools import wraps
from pyspark.sql import DataFrame
from typing import Tuple, TypeVar, Callable

T = TypeVar('T', DataFrame, Tuple[DataFrame, ...])

def repartition(spark: SparkSession):
    def repartition_output(func: Callable[..., T]) -> Callable[..., T]:
        @wraps(func)
        def wrapper(*args, **kwargs) -> T:
            result = func(*args, **kwargs)
                    
            if isinstance(result, DataFrame):
                return result.repartition(spark.sparkContext.defaultParallelism)
            
            elif isinstance(result, tuple):
                return tuple(
                    df.repartition(spark.sparkContext.defaultParallelism) 
                    if isinstance(df, DataFrame) else df 
                    for df in result
                )
            
            return result
        
        return wrapper
    return repartition_output

In [11]:
etl = ETLPipeline(spark)
etl.transform = repartition(spark)(etl.transform)
etl._process_silver_layer = repartition(spark)(etl._process_silver_layer)
etl._process_gold_layer = repartition(spark)(etl._process_gold_layer)

In [12]:
wall_times = []

for _ in tqdm(range(3)):
    create_directories('..')
    start = time.time()
    etl(input_path)
    end = time.time()
    wall_times.append(end - start)
    clean_directories('..')

print(f"Average Wall Time: {(sum(wall_times) / len(wall_times)) * 1000} ms")

  0%|          | 0/3 [00:00<?, ?it/s]

Average Wall Time: 47621.26064300537 ms
